# Broadcast joins vs bucket joins on Databricks

This notebook uses the Halo match datasets in `/Volumes/tabular/dataexpert/mg_volume/` to explain how broadcast joins and bucket joins reduce shuffle during joins.

What you'll learn:
* Why a regular join shuffles both sides
* Why broadcast joins are usually best when one side is small
* What classic bucket joins are trying to achieve
* The Databricks-native alternative on Unity Catalog and Serverless compute

Important environment note:
* True Hive bucket joins are not available on Unity Catalog managed Delta tables on Serverless compute.
* So this notebook compares three practical strategies on the same data:
  1. a baseline shuffle join on raw CSV data
  2. a broadcast join
  3. a Delta-native, bucket-like layout strategy using `CLUSTER BY`, `repartition()`, and `sortWithinPartitions()`

At the end, we'll compare execution plans, runtime, and when each approach is the better choice.


In [0]:
from pyspark.sql import functions as F
import time
import pandas as pd
import matplotlib.pyplot as plt

matches_raw = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/tabular/dataexpert/mg_volume/matches.csv")
)

match_details_raw = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/tabular/dataexpert/mg_volume/match_details.csv")
)

matches_raw.createOrReplaceTempView("raw_matches")
match_details_raw.createOrReplaceTempView("raw_match_details")

print("Loaded raw source files and registered temp views: raw_matches, raw_match_details")


## 2. Create an optimized Delta layout

Classic bucket joins require bucket metadata on both sides of the join. In this environment, we can't use that feature directly.

Instead, we'll create Delta tables that:
* cluster by `match_id`
* write data after `repartition(16, "match_id")`
* sort rows within each partition by `match_id`

This does not create a true bucket join, but it makes the physical layout much closer to what bucketing is trying to achieve: rows with the same join key are stored together.


In [0]:
matches_metrics = matches_raw.agg(
    F.count("*").alias("rows"),
    F.countDistinct("match_id").alias("distinct_matches"),
    F.min("completion_date").alias("min_completion_date"),
    F.max("completion_date").alias("max_completion_date")
).toPandas()

match_details_metrics = match_details_raw.agg(
    F.count("*").alias("rows"),
    F.countDistinct("match_id").alias("distinct_matches"),
    F.countDistinct("player_gamertag").alias("distinct_players"),
    F.avg("player_total_kills").alias("avg_kills"),
    F.avg("player_total_deaths").alias("avg_deaths")
).toPandas()

summary_df = pd.DataFrame([
    {
        "dataset": "matches",
        "rows": int(matches_metrics.loc[0, "rows"]),
        "distinct_matches": int(matches_metrics.loc[0, "distinct_matches"]),
        "date_range": f"{matches_metrics.loc[0, 'min_completion_date'].date()} to {matches_metrics.loc[0, 'max_completion_date'].date()}",
        "distinct_players": None,
        "avg_kills": None,
        "avg_deaths": None,
    },
    {
        "dataset": "match_details",
        "rows": int(match_details_metrics.loc[0, "rows"]),
        "distinct_matches": int(match_details_metrics.loc[0, "distinct_matches"]),
        "date_range": "n/a",
        "distinct_players": int(match_details_metrics.loc[0, "distinct_players"]),
        "avg_kills": round(float(match_details_metrics.loc[0, "avg_kills"]), 2),
        "avg_deaths": round(float(match_details_metrics.loc[0, "avg_deaths"]), 2),
    },
])

display(summary_df)

row_count_df = summary_df[["dataset", "rows"]].copy()
ax = row_count_df.plot(
    kind="bar",
    x="dataset",
    y="rows",
    legend=False,
    figsize=(8, 4),
    color=["#4C78A8", "#F58518"],
    title="Table size comparison"
)
ax.set_ylabel("Row count")
ax.set_xlabel("")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

players_per_match = match_details_metrics.loc[0, "rows"] / match_details_metrics.loc[0, "distinct_matches"]
print(f"match_details has about {players_per_match:.1f} player rows per match.")
print("Because the matches table is much smaller, it is a strong candidate for a broadcast join.")


In [0]:
%sql
DROP TABLE IF EXISTS tabular.dataexpert.mg_matches_bucketed;
DROP TABLE IF EXISTS tabular.dataexpert.mg_match_details_bucketed;

CREATE TABLE tabular.dataexpert.mg_matches_bucketed (
    match_id STRING,
    is_team_game BOOLEAN,
    playlist_id STRING,
    completion_date TIMESTAMP
)
USING DELTA
CLUSTER BY (match_id);

CREATE TABLE tabular.dataexpert.mg_match_details_bucketed (
    match_id STRING,
    player_gamertag STRING,
    player_total_kills INTEGER,
    player_total_deaths INTEGER
)
USING DELTA
CLUSTER BY (match_id);

In [0]:
(
    matches_raw.select("match_id", "is_team_game", "playlist_id", "completion_date")
    .repartition(16, "match_id")
    .sortWithinPartitions("match_id")
    .write.mode("overwrite")
    .saveAsTable("tabular.dataexpert.mg_matches_bucketed")
)

(
    match_details_raw.select("match_id", "player_gamertag", "player_total_kills", "player_total_deaths")
    .repartition(16, "match_id")
    .sortWithinPartitions("match_id")
    .write.mode("overwrite")
    .saveAsTable("tabular.dataexpert.mg_match_details_bucketed")
)

optimized_counts = spark.sql("""
SELECT 'mg_matches_bucketed' AS table_name, COUNT(*) AS rows FROM tabular.dataexpert.mg_matches_bucketed
UNION ALL
SELECT 'mg_match_details_bucketed' AS table_name, COUNT(*) AS rows FROM tabular.dataexpert.mg_match_details_bucketed
""").toPandas()

display(optimized_counts)
print("Optimized Delta tables created successfully.")

## 3. Benchmark three join strategies

We'll run the same aggregation three ways:

* Baseline shuffle join: raw CSV views joined on `match_id`
* Broadcast join: the same query, but with a `BROADCAST` hint on the smaller `matches` table
* Delta-native layout join: join the optimized Delta tables

For each strategy we'll capture:
* elapsed time
* join operator from `EXPLAIN FORMATTED`
* number of `Exchange` operators, which are a good proxy for shuffle work

The output rows should be identical. What should change is the execution strategy.


In [0]:
benchmark_template = """
SELECT {join_hint}
       m.playlist_id,
       COUNT(*) AS joined_rows,
       COUNT(DISTINCT d.player_gamertag) AS distinct_players,
       ROUND(AVG(d.player_total_kills), 2) AS avg_kills,
       ROUND(AVG(d.player_total_deaths), 2) AS avg_deaths
FROM {matches_source} m
JOIN {details_source} d
  ON m.match_id = d.match_id
GROUP BY m.playlist_id
ORDER BY joined_rows DESC, m.playlist_id
LIMIT 10
"""

queries = {
    "Baseline shuffle join": benchmark_template.format(
        join_hint="",
        matches_source="raw_matches",
        details_source="raw_match_details",
    ),
    "Broadcast join": benchmark_template.format(
        join_hint="/*+ BROADCAST(m) */",
        matches_source="raw_matches",
        details_source="raw_match_details",
    ),
    "Delta-native layout join": benchmark_template.format(
        join_hint="",
        matches_source="tabular.dataexpert.mg_matches_bucketed",
        details_source="tabular.dataexpert.mg_match_details_bucketed",
    ),
}

def plan_text_for(query: str) -> str:
    return "\n".join(row[0] for row in spark.sql(f"EXPLAIN FORMATTED {query}").collect())

def detect_join_operator(plan_text: str) -> str:
    for operator in ["BroadcastHashJoin", "SortMergeJoin", "ShuffledHashJoin", "BroadcastNestedLoopJoin"]:
        if operator in plan_text:
            return operator
    return "Unknown"

def extract_plan_highlights(plan_text: str) -> str:
    interesting = []
    for line in plan_text.splitlines():
        if any(token in line for token in ["BroadcastHashJoin", "SortMergeJoin", "Exchange", "Scan csv", "Scan ExistingRDD", "Scan parquet", "Scan Delta"]):
            interesting.append(line.strip())
    return "\n".join(interesting[:10])

def run_benchmark(name: str, query: str) -> dict:
    plan_text = plan_text_for(query)
    start = time.perf_counter()
    result_pdf = spark.sql(query).toPandas()
    elapsed_seconds = time.perf_counter() - start
    return {
        "strategy": name,
        "elapsed_seconds": round(elapsed_seconds, 3),
        "join_operator": detect_join_operator(plan_text),
        "exchange_operators": plan_text.count("Exchange"),
        "plan_highlights": extract_plan_highlights(plan_text),
        "result_pdf": result_pdf,
    }

benchmark_runs = [run_benchmark(name, query) for name, query in queries.items()]
result_frames = {run["strategy"]: run["result_pdf"] for run in benchmark_runs}

reference_pdf = (
    result_frames["Broadcast join"]
    .sort_values(["joined_rows", "playlist_id"], ascending=[False, True])
    .reset_index(drop=True)
)

benchmark_df = pd.DataFrame([
    {
        "strategy": run["strategy"],
        "elapsed_seconds": run["elapsed_seconds"],
        "join_operator": run["join_operator"],
        "exchange_operators": run["exchange_operators"],
        "same_output_as_broadcast": reference_pdf.equals(
            run["result_pdf"].sort_values(["joined_rows", "playlist_id"], ascending=[False, True]).reset_index(drop=True)
        ),
    }
    for run in benchmark_runs
]).sort_values("elapsed_seconds").reset_index(drop=True)

display(benchmark_df)

print("Plan highlights by strategy:\n")
for run in benchmark_runs:
    print(f"{run['strategy']}:\n{run['plan_highlights']}\n")

print("Top 10 playlist-level results (from the broadcast join query):")
display(result_frames["Broadcast join"])


In [0]:
plot_df = benchmark_df.copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(
    plot_df["strategy"],
    plot_df["elapsed_seconds"],
    color=["#4C78A8", "#F58518", "#54A24B"]
)
axes[0].set_title("Elapsed time by strategy")
axes[0].set_ylabel("Seconds")
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar(
    plot_df["strategy"],
    plot_df["exchange_operators"],
    color=["#4C78A8", "#F58518", "#54A24B"]
)
axes[1].set_title("Shuffle-related Exchange operators")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

fastest = benchmark_df.iloc[0]
print(
    f"Fastest strategy in this run: {fastest['strategy']} "
    f"({fastest['elapsed_seconds']} seconds, {fastest['join_operator']})."
)
print("Broadcast joins usually win when one side is small enough to ship to every worker.")
print("The Delta-native layout strategy helps storage layout, but it does not create a true bucket join on Serverless.")

## 4. What this tutorial shows

### Broadcast join
* Best when one side of the join is small enough to replicate to every worker.
* Avoids shuffling the large side of the join.
* In this dataset, `matches` is much smaller than `match_details`, so broadcasting `matches` is the strongest strategy.

### Classic bucket join
* Best when both sides are large, reused often, and bucketed on the same join key with matching bucket counts.
* Reduces shuffle by pre-arranging both tables on disk.
* Useful when broadcasting is not possible because neither side is small.

### Databricks-native alternative in this notebook
* `CLUSTER BY (match_id)` improves file layout over time.
* `repartition(16, "match_id")` and `sortWithinPartitions("match_id")` make the write path more join-friendly.
* This improves locality, but it is still not the same thing as a true bucket join.

### Practical takeaway
* If one side is small: use a broadcast join.
* If both sides are large on Databricks: prefer Delta layout optimizations such as liquid clustering and good join keys.
* If you truly need Hive-style bucket joins for a demo, you need a non-Serverless environment with external Parquet or ORC tables.
